# 02 - Data Collection
Fetch Polymarket probability timeseries and freight index data.

In [ ]:
import sys
sys.path.insert(0, "..")
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'white'

## Step 1: Load discovered markets

In [ ]:
from src.polymarket.market_discovery import load_discovered_markets

markets_df = load_discovered_markets()
print(f'Loaded {len(markets_df)} markets')
markets_df[['market_id', 'title', 'category', 'status']].head(10)

## Step 2: Fetch Polymarket timeseries

In [ ]:
from src.polymarket.client import PolymarketClient
from src.polymarket.timeseries import TimeseriesFetcher

client = PolymarketClient()
fetcher = TimeseriesFetcher(client)
timeseries = fetcher.fetch_all(markets_df)
print(f'Fetched timeseries for {len(timeseries)} markets')

In [ ]:
# Show coverage statistics
coverages = []
for mid, df in timeseries.items():
    if not df.empty:
        coverages.append({
            'market_id': mid,
            'start': df['date'].min(),
            'end': df['date'].max(),
            'n_days': len(df),
            'title': df['market_title'].iloc[0] if 'market_title' in df.columns else mid,
        })
coverage_df = pd.DataFrame(coverages)
print('Data coverage summary:')
coverage_df.describe()

## Step 3: Visualize key Polymarket timeseries

In [ ]:
# Plot probability curves for first 6 markets with data
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()
plotted = 0
for mid, df in list(timeseries.items())[:6]:
    ax = axes[plotted]
    df_plot = df.copy()
    df_plot['date'] = pd.to_datetime(df_plot['date'])
    ax.plot(df_plot['date'], df_plot['probability'] * 100, color='#1f6aa5', linewidth=1.5)
    ax.fill_between(df_plot['date'], 0, df_plot['probability'] * 100,
                    alpha=0.1, color='#1f6aa5')
    title = df['market_title'].iloc[0] if 'market_title' in df.columns else mid
    ax.set_title(title[:55], fontsize=9, fontweight="bold")
    ax.set_ylabel('Probability (%)')
    ax.set_ylim(0, 105)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plotted += 1

plt.suptitle('Polymarket Probability Timeseries - Key Supply Chain Markets',
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig('../output/figures/polymarket_timeseries_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Fetch freight index data

In [ ]:
from src.freight.scraper import fetch_all_freight_indexes, print_download_instructions
from src.freight.normalize import prepare_freight_panel

print_download_instructions()
freight_raw = fetch_all_freight_indexes(use_synthetic_fallback=True)
print(f'
Loaded freight indexes: {list(freight_raw.keys())}')

In [ ]:
freight_processed = prepare_freight_panel(freight_raw)

## Step 5: Visualize freight data

In [ ]:
fig, axes = plt.subplots(len(freight_processed), 1,
                         figsize=(12, 4 * len(freight_processed)))
if len(freight_processed) == 1:
    axes = [axes]
for ax, (name, df) in zip(axes, freight_processed.items()):
    df_plot = df.copy()
    df_plot['date'] = pd.to_datetime(df_plot['date'])
    ax.plot(df_plot['date'], df_plot['value'], color='#e07b39', linewidth=1.8)
    ax.fill_between(df_plot['date'], df_plot['value'].min() * 0.95,
                    df_plot['value'], alpha=0.08, color='#e07b39')
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_ylabel('Index Value')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
plt.suptitle('Freight Index Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/figures/freight_indexes.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary
Data collection complete. Proceed to 03_analysis.ipynb.